## CUDA Kernel Optimization Using Genetic Algorithms And LLMs: 
&nbsp;&nbsp;&nbsp; In this notebook, we will explore an approach to optimizing CUDA kernels using an intersection of both Genetic Algorithms and LLMs. <br> 
&nbsp;&nbsp;&nbsp; To speed up the whole process, we will rely on a set of kernels, and we will try to measure the optimization on them, accounting for different configurations that we will set. 


### Dataset Exploration : 
&nbsp;&nbsp;&nbsp; In this section of the notebook, we will analyze the overall dataset and the operation that it has so that we can select a subset of operations that we will be optimizing in this approach rather than focusing on all of the operations.<br> 

In [1]:
from datasets import load_dataset
import os
import re
import json
import time
import copy
import random
import uuid
import math
import pandas as pd
from dataclasses import dataclass, asdict, field
from typing import Optional
dataset_tag = "SakanaAI/AI-CUDA-Engineer-Archive"
ds = load_dataset("SakanaAI/AI-CUDA-Engineer-Archive", split="level_1")
df = ds.to_pandas()
cuda_col = "CUDA_Code"
TARGET_MATRIX_SIZE = 512
# filtering for correct kernels only 
filtered = df[
        (df["Correct"] == True) &
        (df[cuda_col].notna()) &
        (df[cuda_col].str.strip() != "")].copy()
print(f"The total number of correct kernels is {len(filtered)}")

The total number of correct kernels is 7222


In [2]:
print(f"The columns of the dataset are as follows : {filtered.columns}")

The columns of the dataset are as follows : Index(['Op_Name', 'Level_ID', 'Task_ID', 'Kernel_Name', 'CUDA_Runtime',
       'PyTorch_Native_Runtime', 'PyTorch_Compile_Runtime',
       'CUDA_Speedup_Native', 'CUDA_Speedup_Compile', 'CUDA_Code',
       'PyTorch_Code_Module', 'PyTorch_Code_Functional', 'Correct', 'Max_Diff',
       'Error', 'NCU_Profile', 'Torch_Profile', 'Clang_Tidy',
       '__index_level_0__'],
      dtype='object')


In [3]:
print(f"The number of operation covered by this dataset is as follows:\n{filtered['Op_Name'].value_counts()}")

The number of operation covered by this dataset is as follows:
Op_Name
98_KLDivLoss                                                                          416
15_Matmul_for_lower_triangular_matrices                                               309
42_Max_Pooling_2D                                                                     307
30_Softsign                                                                            98
31_ELU                                                                                 97
                                                                                     ... 
72_conv_transposed_3D_asymmetric_input_asymmetric_kernel___strided_padded_grouped_     51
6_Matmul_with_large_K_dimension_                                                       45
60_conv_standard_3D__square_input__asymmetric_kernel                                   45
58_conv_transposed_3D__asymmetric_input__asymmetric_kernel                             42
87_conv_pointwise_2D         

In [4]:
for op, count in filtered['Op_Name'].value_counts().items():
    print(f"The operation is {op} and it has {count} kernels.")

The operation is 98_KLDivLoss and it has 416 kernels.
The operation is 15_Matmul_for_lower_triangular_matrices and it has 309 kernels.
The operation is 42_Max_Pooling_2D and it has 307 kernels.
The operation is 30_Softsign and it has 98 kernels.
The operation is 31_ELU and it has 97 kernels.
The operation is 5_Matrix_scalar_multiplication and it has 97 kernels.
The operation is 12_Matmul_with_diagonal_matrices_ and it has 96 kernels.
The operation is 28_HardSigmoid and it has 93 kernels.
The operation is 21_Sigmoid and it has 93 kernels.
The operation is 20_LeakyReLU and it has 93 kernels.
The operation is 97_CosineSimilarityLoss and it has 92 kernels.
The operation is 88_MinGPTNewGelu and it has 88 kernels.
The operation is 96_HuberLoss and it has 88 kernels.
The operation is 94_MSELoss and it has 87 kernels.
The operation is 27_SELU_ and it has 86 kernels.
The operation is 38_L1Norm_ and it has 86 kernels.
The operation is 50_Product_reduction_over_a_dimension and it has 86 kernels.


#### Remark : 
We can clearly see that we have a lot of operations, and that is why we will restrict them to kernels that have significance and that have enough kernels in terms of numbers so that we can introduce diversity in our initial population and further propagate it. <br> 
Therefor, the picked kernels are as follows : 
- matrix multiplication.
- MHA.
- convolution.

In [ ]:
import random
from dataclasses import dataclass, asdict

@dataclass
class ExecutionPlan:
   
    tile_size: int = 16          
    block_x: int = 16            
    block_y: int = 16            
    unroll: int = 4              
    vector_width: int = 1        
    num_stages: int = 2         

   
    matrix_size: int = 512       
    use_launch_bounds: bool = False
    launch_bounds_max_threads: int = 256
    launch_bounds_min_blocks: int = 2

    pad_shared: bool = False     
    use_ldg: bool = False        
    use_cp_async: bool = False  
    use_double_buffer: bool = False  
    use_wmma: bool = False    
    warps_x: int = 4      # new
    warps_y: int = 2
    
    TILE_SIZES = [8, 16, 32, 64]
    BLOCK_DIMS = [8, 16, 32, 64, 128, 256]
    UNROLL_VALS = [1, 2, 4, 8]
    VECTOR_WIDTHS = [1, 2, 4]
    NUM_STAGES = [1, 2, 3, 4]
    MATRIX_SIZES = [128, 256, 512, 1024, 2048]
    LAUNCH_BOUNDS_MAX = [128, 256, 512, 1024]
    LAUNCH_BOUNDS_MIN = [1, 2, 4]
    WARPS_X_VALS = [2, 4, 8]
    WARPS_Y_VALS = [1, 2, 4]

    def __post_init__(self):
        self.validate_and_fix()

    def validate_and_fix(self):
        
        self.tile_size = self._nearest(self.tile_size, self.TILE_SIZES)
        self.block_x = self._nearest(self.block_x, self.BLOCK_DIMS)
        self.block_y = self._nearest(self.block_y, self.BLOCK_DIMS)
        self.unroll = self._nearest(self.unroll, self.UNROLL_VALS)
        self.vector_width = self._nearest(self.vector_width, self.VECTOR_WIDTHS)
        self.num_stages = self._nearest(self.num_stages, self.NUM_STAGES)
        self.matrix_size = self._nearest(self.matrix_size, self.MATRIX_SIZES)

       
        while self.block_x * self.block_y > 1024:
            if self.block_x >= self.block_y:
                idx = self.BLOCK_DIMS.index(self.block_x)
                self.block_x = self.BLOCK_DIMS[max(0, idx - 1)]
            else:
                idx = self.BLOCK_DIMS.index(self.block_y)
                self.block_y = self.BLOCK_DIMS[max(0, idx - 1)]

     
        if self.tile_size > self.matrix_size:
            self.tile_size = min(self.TILE_SIZES, key=lambda x: abs(x - self.matrix_size))

       
        if self.use_launch_bounds:
            self.launch_bounds_max_threads = self._nearest(self.launch_bounds_max_threads, self.LAUNCH_BOUNDS_MAX)
            self.launch_bounds_min_blocks = self._nearest(self.launch_bounds_min_blocks, self.LAUNCH_BOUNDS_MIN)
            # Ensure max_threads at least block_x * block_y
            block_product = self.block_x * self.block_y
            if self.launch_bounds_max_threads < block_product:
                self.launch_bounds_max_threads = block_product
        # Always validate warps_x and warps_y (not just when use_wmma=True)
        self.warps_x = self._nearest(self.warps_x, self.WARPS_X_VALS)
        self.warps_y = self._nearest(self.warps_y, self.WARPS_Y_VALS)
        
        if self.use_wmma:
            if self.tile_size % 16 != 0:
                self.tile_size = 16

    @staticmethod
    def _nearest(value, options):
        return min(options, key=lambda x: abs(x - value))

    def to_dict(self) -> dict:
        return asdict(self)

    def to_defines(self) -> str:
        """Render all performance parameters as C preprocessor directives."""
        lines = [
            f"#define TILE_SIZE {self.tile_size}",
            f"#define BLOCK_SIZE {self.tile_size}",  # Set BLOCK_SIZE = TILE_SIZE by default
            f"#define BLOCK_X {self.block_x}",
            f"#define BLOCK_Y {self.block_y}",
            f"#define UNROLL {self.unroll}",
            f"#define VECTOR_WIDTH {self.vector_width}",
            f"#define NUM_STAGES {self.num_stages}",
        ]
        if self.use_launch_bounds:
            lines.append("#define USE_LAUNCH_BOUNDS 1")
            lines.append(f"#define LAUNCH_BOUNDS_MAX_THREADS {self.launch_bounds_max_threads}")
            lines.append(f"#define LAUNCH_BOUNDS_MIN_BLOCKS {self.launch_bounds_min_blocks}")
        if self.pad_shared:
            lines.append("#define PAD_SHARED 1")
        if self.use_ldg:
            lines.append("#define USE_LDG 1")
        if self.use_cp_async:
            lines.append("#define USE_CP_ASYNC 1")
        if self.use_double_buffer:
            lines.append("#define USE_DOUBLE_BUFFER 1")
        if self.use_wmma:
            lines.append("#define USE_WMMA 1")
        if self.use_wmma:
            lines.append(f"#define WARPS_X {self.warps_x}")
            lines.append(f"#define WARPS_Y {self.warps_y}")
        return "\n".join(lines)

    @classmethod
    def random(cls) -> "ExecutionPlan":
        """Create a random legal plan for initialisation."""
        return cls(
            tile_size=random.choice(cls.TILE_SIZES),
            block_x=random.choice(cls.BLOCK_DIMS),
            block_y=random.choice(cls.BLOCK_DIMS),
            unroll=random.choice(cls.UNROLL_VALS),
            vector_width=random.choice(cls.VECTOR_WIDTHS),
            num_stages=random.choice(cls.NUM_STAGES),
            matrix_size=random.choice(cls.MATRIX_SIZES),
            use_launch_bounds=random.choice([True, False]),
            launch_bounds_max_threads=random.choice(cls.LAUNCH_BOUNDS_MAX),
            launch_bounds_min_blocks=random.choice(cls.LAUNCH_BOUNDS_MIN),
            pad_shared=random.choice([True, False]),
            use_ldg=random.choice([True, False]),
            use_cp_async=random.choice([True, False]),
            use_double_buffer=random.choice([True, False]),
            use_wmma=random.choice([True, False]),
            warps_x=random.choice(cls.WARPS_X_VALS),
            warps_y=random.choice(cls.WARPS_Y_VALS),
        )

    @classmethod
    def from_dict(cls, d: dict) -> "ExecutionPlan":
        """Construct from dictionary (e.g., from Qwen output)."""
        return cls(
            tile_size=d.get("tile_size", 16),
            block_x=d.get("block_x", 16),
            block_y=d.get("block_y", 16),
            unroll=d.get("unroll", 4),
            vector_width=d.get("vector_width", 1),
            num_stages=d.get("num_stages", 2),
            matrix_size= TARGET_MATRIX_SIZE,
            use_launch_bounds=d.get("use_launch_bounds", False),
            launch_bounds_max_threads=d.get("launch_bounds_max_threads", 256),
            launch_bounds_min_blocks=d.get("launch_bounds_min_blocks", 2),
            pad_shared=d.get("pad_shared", False),
            use_ldg=d.get("use_ldg", False),
            use_cp_async=d.get("use_cp_async", False),
            use_double_buffer=d.get("use_double_buffer", False),
            use_wmma=d.get("use_wmma", False),
            warps_x=d.get("warps_x", 1),
            warps_y=d.get("warps_y", 1),
        )

In [6]:
def get_extra_defines(op_name: str) -> dict:
    extra = {}
    op = op_name.lower()
    if "transposed_a" in op:
        extra["TRANS_A"] = 1
    if "transposed_b" in op:
        extra["TRANS_B"] = 1
    if "batch" in op and "matmul" in op:
        extra["BATCHED"] = 1
    if "causal" in op or "mask" in op:
        extra["CAUSAL_MASK"] = 1
    return extra

In [7]:
DEFINE_GUARDS = """\
#ifndef TILE_SIZE
#define TILE_SIZE 32
#endif
#ifndef BLOCK_SIZE
#define BLOCK_SIZE TILE_SIZE
#endif
#ifndef BLOCK_X
#define BLOCK_X 16
#endif
#ifndef BLOCK_Y
#define BLOCK_Y 16
#endif
#ifndef UNROLL
#define UNROLL 4
#endif
#ifndef VECTOR_WIDTH
#define VECTOR_WIDTH 1
#endif
#ifndef NUM_STAGES
#define NUM_STAGES 2
#endif
"""

def inject_parameters(static_code: str, plan: ExecutionPlan, extra_defines: dict = None) -> str:
    # 1. Strip ALL managed defines from the kernel body
    clean = strip_existing_defines(static_code)
    
    # 2. Build GA defines
    defines = plan.to_defines()
    if extra_defines:
        extra   = "\n".join(f"#define {k} {v}" for k, v in extra_defines.items())
        defines = extra + "\n" + defines
    
    # 3. Assemble: GA defines first (priority), then guards (fallback), then clean kernel body
    return defines + "\n\n" + DEFINE_GUARDS + "\n" + clean

In [8]:
# Sanityy checkkk 
ep = ExecutionPlan(tile_size=16, block_x=32, block_y=32, unroll=4)  # 32*32=1024 — legal
bad = ExecutionPlan(tile_size=16, block_x=32, block_y=64)            # 32*64=2048 — auto-fixed
print(f"Legal plan:   {ep.to_dict()}")
print(f"Auto-fixed:   {bad.to_dict()}  (block product={bad.block_x*bad.block_y})")
print(f"Defines:\n{ep.to_defines()}")

Legal plan:   {'tile_size': 16, 'block_x': 32, 'block_y': 32, 'unroll': 4, 'vector_width': 1, 'num_stages': 2, 'matrix_size': 512, 'use_launch_bounds': False, 'launch_bounds_max_threads': 256, 'launch_bounds_min_blocks': 2, 'pad_shared': False, 'use_ldg': False, 'use_cp_async': False, 'use_double_buffer': False, 'use_wmma': False, 'warps_x': 4, 'warps_y': 2}
Auto-fixed:   {'tile_size': 16, 'block_x': 32, 'block_y': 32, 'unroll': 4, 'vector_width': 1, 'num_stages': 2, 'matrix_size': 512, 'use_launch_bounds': False, 'launch_bounds_max_threads': 256, 'launch_bounds_min_blocks': 2, 'pad_shared': False, 'use_ldg': False, 'use_cp_async': False, 'use_double_buffer': False, 'use_wmma': False, 'warps_x': 4, 'warps_y': 2}  (block product=1024)
Defines:
#define TILE_SIZE 16
#define BLOCK_X 32
#define BLOCK_Y 32
#define UNROLL 4
#define VECTOR_WIDTH 1
#define NUM_STAGES 2


## Model Loading : 
&nbsp;&nbsp;&nbsp; In here, we will go ahead and load a Qwen model (7B), this model has a crucial importance at 2 steps within the genetic algorithm that are : 
- **crossover**: The model will take the execution model of the two parents and apply the crossover and that is by identifying a much better configuration using the values of the parents.
- **mutation**: Random change is introduced at the level of the execution plan configuration. 

In [9]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print("Model ready.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 NVL MIG 1g.24gb. Num GPUs = 1. Max memory: 21.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model ready.


In [10]:
import json
import re

def _parse_plan_from_response(response: str, fallback) -> "ExecutionPlan":
    """
    Extract the first complete JSON object from Qwen's response and parse it.
    Handles:
      - Responses wrapped in ```json ... ```
      - Leading/trailing text
      - Trailing commas (by using a simple regex removal)
      - Nested braces
    """
    # Remove markdown code fences
    text = re.sub(r'```json\s*', '', response)
    text = re.sub(r'```\s*$', '', text)
    text = text.strip()

    # Find the first '{' and then match braces counting nesting
    start = text.find('{')
    if start == -1:
        print(f"  [Qwen] No JSON object found. Response: {response[:200]}")
        return copy.deepcopy(fallback)

    # Count braces to find matching closing brace
    brace_count = 0
    end = start
    for i, ch in enumerate(text[start:], start=start):
        if ch == '{':
            brace_count += 1
        elif ch == '}':
            brace_count -= 1
            if brace_count == 0:
                end = i
                break
    if brace_count != 0:
        print(f"  [Qwen] Unbalanced braces in JSON. Response: {text[:200]}")
        return copy.deepcopy(fallback)

    json_str = text[start:end+1]
    # Remove trailing commas (common mistake by LLMs)
    json_str = re.sub(r',\s*}', '}', json_str)
    json_str = re.sub(r',\s*]', ']', json_str)

    try:
        data = json.loads(json_str)
        plan = ExecutionPlan.from_dict(data)
        print(f"  [Qwen] Parsed plan: {plan.to_dict()}")
        return plan
    except Exception as e:
        print(f"  [Qwen] JSON parse error ({e}). Raw JSON: {json_str[:200]}")
        return copy.deepcopy(fallback)


In [11]:
def _call_qwen_for_plan(prompt: str, fallback) -> "ExecutionPlan":
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=512,   # increased from 256
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
        )

    response = tokenizer.batch_decode(
        outputs[:, inputs.shape[1]:], skip_special_tokens=True
    )[0].strip()

    print(f"  [Qwen] Raw response (first 500 chars): {response[:500]}...")
    return _parse_plan_from_response(response, fallback)

In [12]:
def _detect_used_defines(static_code: str) -> list:
    used = []
    for name in sorted(_MANAGED_DEFINES):
        if re.search(r'\b' + name + r'\b', static_code):
            used.append(name)
    return used

In [13]:
# TODO: improve the prompt 
def qwen_crossover(
    static_code : str,
    parent1     : "ExecutionPlan",
    parent2     : "ExecutionPlan",
    speedup1    : float,
    speedup2    : float,
    op_name : str 
) -> "ExecutionPlan":
    """
    Blend two parent ExecutionPlans into a child, guided by the kernel structure.
    """
    used_defines   = _detect_used_defines(static_code)
    kernel_type = normalize_op_name(op_name)

    type_hint = (
        "OPTIMIZATION FOCUS BY KERNEL TYPE:\n"
            "OPTIMIZATION FOCUS BY KERNEL TYPE:\n"
            "matmul -> STRONGLY prefer use_wmma=true on sm_90 (H100). "
            "WMMA uses tensor cores and gives 4-8x throughput over FP32. "
            "When use_wmma=true, tile_size must be a multiple of 16, "
            "block_x and block_y must satisfy block_x*block_y >= 256.\n"
        "  matmul      -> tiling, occupancy, memory coalescing, shared-memory reuse\n"
        "  convolution -> spatial reuse, stride/padding awareness, group handling, bank-conflict avoidance\n"
        "  softmax     -> reduction efficiency, numerically stable normalization, warp-friendly execution\n"
        "  unknown     -> choose conservative, generally valid settings\n"
    )

    define_hint  = (
        f"This kernel uses these #defines from your plan: {used_defines}.\n"
        f"Unused fields still affect future children — include all fields in your JSON."
    ) if used_defines else ""

    prompt = f"""You are a CUDA performance-tuning expert.a 
    

KERNEL TYPE:
{kernel_type}

{type_hint}
KERNEL STRUCTURE (first 600 chars):
{static_code[:600]}

{define_hint}

PARENT A (speedup={speedup1:.2f}x):
{json.dumps(parent1.to_dict())}

PARENT B (speedup={speedup2:.2f}x):
{json.dumps(parent2.to_dict())}

FULL EXECUTION PLAN FIELDS:
  tile_size                : {ExecutionPlan.TILE_SIZES}
  block_x                  : {ExecutionPlan.BLOCK_DIMS}
  block_y                  : {ExecutionPlan.BLOCK_DIMS}
  unroll                   : {ExecutionPlan.UNROLL_VALS}
  vector_width             : {ExecutionPlan.VECTOR_WIDTHS}
  num_stages               : {ExecutionPlan.NUM_STAGES}
  use_launch_bounds        : [0, 1]
  launch_bounds_max_threads: {ExecutionPlan.LAUNCH_BOUNDS_MAX}
  launch_bounds_min_blocks : {ExecutionPlan.LAUNCH_BOUNDS_MIN}
  pad_shared               : [0, 1]
  use_ldg                  : [0, 1]
  use_cp_async             : [0, 1]
  use_double_buffer        : [0, 1]
  use_wmma                 : [0, 1]
  warps_x                  : {ExecutionPlan.WARPS_X_VALS}
  warps_y                  : {ExecutionPlan.WARPS_Y_VALS}

LEGAL VALUES:
  tile_size    : {ExecutionPlan.TILE_SIZES}
  block_x      : {ExecutionPlan.BLOCK_DIMS}   block_y : {ExecutionPlan.BLOCK_DIMS}
  CONSTRAINT   : block_x * block_y <= 1024
  unroll       : {ExecutionPlan.UNROLL_VALS}
  vector_width : {ExecutionPlan.VECTOR_WIDTHS}
  num_stages   : {ExecutionPlan.NUM_STAGES}
  wraps_x      : {ExecutionPlan.WARPS_X_VALS}
    wraps_y      : {ExecutionPlan.WARPS_Y_VALS}

TASK: Combine the best traits of both parents into ONE child plan that is
likely to achieve higher speedup. Match the tuning strategy to the kernel
type, and consider occupancy, memory coalescing, shared-memory bank conflicts,
launch bounds, and memory-transfer flags.

CRITICAL: Return ONLY a JSON object with integer values. Do NOT include any
#define lines, code, or explanation. Example:
{{"tile_size": 32, "block_x": 16, "block_y": 16, "unroll": 4, "vector_width": 4, "num_stages": 2, "use_launch_bounds": 0, "launch_bounds_max_threads": 256, "launch_bounds_min_blocks": 2, "pad_shared": 0, "use_ldg": 1, "use_cp_async": 0, "use_double_buffer": 0, "use_wmma": 0 , "warps_x": 4, "warps_y": 2}}"""

    fallback = parent1 if speedup1 >= speedup2 else parent2
    return _call_qwen_for_plan(prompt, fallback)


In [14]:
def qwen_mutate(
    static_code    : str,
    current_plan   : "ExecutionPlan",
    current_speedup: float,
    generation     : int,
    op_name :str , 
) -> "ExecutionPlan":
    used_defines   = _detect_used_defines(static_code)
    kernel_type = normalize_op_name(op_name)

    exploration    = "Explore aggressively — change 2–3 fields" if generation < 3 else "Make targeted small changes — change 1–2 fields"

    define_context = ""
    if used_defines:
        define_context = (
            f"Active #defines in this kernel: {used_defines}.\n"
            f"Focus mutations on fields that map to those defines first.\n"
        )
        if "TILE_SIZE" in used_defines:
            define_context += "TILE_SIZE controls shared-memory tile — larger tiles increase reuse but raise register pressure.\n"
        if "BLOCK_X" in used_defines or "BLOCK_Y" in used_defines:
            define_context += "BLOCK_X/BLOCK_Y set thread block dimensions — ensure block_x*block_y is a multiple of 32 (warp size).\n"
        if "UNROLL" in used_defines:
            define_context += "UNROLL controls loop unrolling — higher values reduce branch overhead but increase register usage.\n"
        if "VECTOR_WIDTH" in used_defines:
            define_context += "VECTOR_WIDTH=4 enables float4 loads (128-bit), which doubles memory throughput if addresses are aligned.\n"

    type_hint = (
        "OPTIMIZATION FOCUS BY KERNEL TYPE:\n"
        "  matmul      -> tiling, occupancy, memory coalescing, shared-memory reuse\n"
        "  convolution -> spatial reuse, stride/padding awareness, group handling, bank-conflict avoidance\n"
        "  softmax     -> reduction efficiency, numerically stable normalization, warp-friendly execution\n"
        "  unknown     -> choose conservative, generally valid settings\n"
    )

    prompt = f"""You are a CUDA performance-tuning expert.
KERNEL TYPE:
{kernel_type}

{type_hint}
KERNEL STRUCTURE (first 600 chars):
{static_code[:600]}

{define_context}
CURRENT PLAN (speedup={current_speedup:.2f}x, generation={generation}):
{json.dumps(current_plan.to_dict())}

FULL EXECUTION PLAN FIELDS:
  tile_size                : {ExecutionPlan.TILE_SIZES}
  block_x                  : {ExecutionPlan.BLOCK_DIMS}
  block_y                  : {ExecutionPlan.BLOCK_DIMS}
  unroll                   : {ExecutionPlan.UNROLL_VALS}
  vector_width             : {ExecutionPlan.VECTOR_WIDTHS}
  num_stages               : {ExecutionPlan.NUM_STAGES}
  matrix_size              : {ExecutionPlan.MATRIX_SIZES}
  use_launch_bounds        : [0, 1]
  launch_bounds_max_threads: {ExecutionPlan.LAUNCH_BOUNDS_MAX}
  launch_bounds_min_blocks : {ExecutionPlan.LAUNCH_BOUNDS_MIN}
  pad_shared               : [0, 1]
  use_ldg                  : [0, 1]
  use_cp_async             : [0, 1]
  use_double_buffer        : [0, 1]
  use_wmma                 : [0, 1]
    warps_x                  : {ExecutionPlan.WARPS_X_VALS}
    warps_y                  : {ExecutionPlan.WARPS_Y_VALS}

LEGAL VALUES:
  tile_size    : {ExecutionPlan.TILE_SIZES}
  block_x      : {ExecutionPlan.BLOCK_DIMS}   block_y : {ExecutionPlan.BLOCK_DIMS}
  CONSTRAINT   : block_x * block_y <= 1024
  unroll       : {ExecutionPlan.UNROLL_VALS}
  vector_width : {ExecutionPlan.VECTOR_WIDTHS}
  num_stages   : {ExecutionPlan.NUM_STAGES}
    wraps_x      : {ExecutionPlan.WARPS_X_VALS}
    wraps_y      : {ExecutionPlan.WARPS_Y_VALS}

STRATEGY: {exploration}

CRITICAL: Return ONLY a JSON object with integer values. Do NOT include any
#define lines, code, or explanation. Example:
{{"tile_size": 32, "block_x": 16, "block_y": 16, "unroll": 4, "vector_width": 4, "num_stages": 2, "matrix_size": 512, "use_launch_bounds": 0, "launch_bounds_max_threads": 256, "launch_bounds_min_blocks": 2, "pad_shared": 0, "use_ldg": 1, "use_cp_async": 0, "use_double_buffer": 0, "use_wmma": 0 , "warps_x": 4, "warps_y": 2}}"""

    return _call_qwen_for_plan(prompt, current_plan)


In [15]:
def arithmetic_crossover(p1: "ExecutionPlan", p2: "ExecutionPlan") -> "ExecutionPlan":
    return ExecutionPlan(
        tile_size = random.choice([p1.tile_size, p2.tile_size]),
        block_x = random.choice([p1.block_x, p2.block_x]),
        block_y = random.choice([p1.block_y, p2.block_y]),
        unroll = random.choice([p1.unroll, p2.unroll]),
        vector_width = random.choice([p1.vector_width, p2.vector_width]),
        num_stages = random.choice([p1.num_stages, p2.num_stages]),
        matrix_size = random.choice([p1.matrix_size, p2.matrix_size]),
        use_launch_bounds = random.choice([p1.use_launch_bounds, p2.use_launch_bounds]),
        launch_bounds_max_threads = random.choice([p1.launch_bounds_max_threads, p2.launch_bounds_max_threads]),
        launch_bounds_min_blocks = random.choice([p1.launch_bounds_min_blocks, p2.launch_bounds_min_blocks]),
        pad_shared = random.choice([p1.pad_shared, p2.pad_shared]),
        use_ldg = random.choice([p1.use_ldg, p2.use_ldg]),
        use_cp_async = random.choice([p1.use_cp_async, p2.use_cp_async]),
        use_double_buffer = random.choice([p1.use_double_buffer, p2.use_double_buffer]),
        use_wmma = random.choice([p1.use_wmma, p2.use_wmma]),
        warps_x = random.choice([p1.warps_x, p2.warps_x]),
        warps_y = random.choice([p1.warps_y, p2.warps_y])
    )





In [16]:

def random_mutation(plan: "ExecutionPlan") -> "ExecutionPlan":
    new = copy.deepcopy(plan)
    field = random.choice([
        "tile_size", "block_x", "block_y", "unroll",
        "vector_width", "num_stages", "matrix_size",
        "use_launch_bounds", "launch_bounds_max_threads", "launch_bounds_min_blocks",
        "pad_shared", "use_ldg", "use_cp_async", "use_double_buffer", "use_wmma",
        "warps_x", "warps_y",
    ])
    opts = {
        "tile_size": ExecutionPlan.TILE_SIZES,
        "block_x": ExecutionPlan.BLOCK_DIMS,
        "block_y": ExecutionPlan.BLOCK_DIMS,
        "unroll": ExecutionPlan.UNROLL_VALS,
        "vector_width": ExecutionPlan.VECTOR_WIDTHS,
        "num_stages": ExecutionPlan.NUM_STAGES,
        "matrix_size": ExecutionPlan.MATRIX_SIZES,
        "use_launch_bounds": [True, False],
        "launch_bounds_max_threads": ExecutionPlan.LAUNCH_BOUNDS_MAX,
        "launch_bounds_min_blocks": ExecutionPlan.LAUNCH_BOUNDS_MIN,
        "pad_shared": [True, False],
        "use_ldg": [True, False],
        "use_cp_async": [True, False],
        "use_double_buffer": [True, False],
        "use_wmma": [True, False],
        "warps_x": ExecutionPlan.WARPS_X_VALS,
        "warps_y": ExecutionPlan.WARPS_Y_VALS,
    }
    setattr(new, field, random.choice(opts[field]))
    new.validate_and_fix()
    return new


## Individual And Population Helpers: 
&nsbp;&nbsp;&nbsp; In this section of the notebook, we will go ahead and define certain helper function required for individual and population. 

In [17]:
# ── GA hyper-parameters ────────────────────────────────────────────────────────
POP_SIZE        = 8      # individuals per generation
GENERATIONS     = 6      # number of GA generations
ELITE_SIZE      = 2      # top-N carried over unchanged
CROSSOVER_PROB  = 0.7    # probability crossover is applied to each child
MUTATION_PROB   = 0.5 # probability mutation is applied to each child
TOURNAMENT_K    = 3      # tournament selection size
RANDOM_SEED     = 42
DATASET_SPLIT   = "level_1"

In [ ]:
import re

# All defines your GA manages — extend this list if you add more
_MANAGED_DEFINES = {
    "TILE_SIZE", "BLOCK_X", "BLOCK_Y", "BLOCK_SIZE",
    "UNROLL", "VECTOR_WIDTH", "NUM_STAGES",
    "USE_LAUNCH_BOUNDS", "LAUNCH_BOUNDS_MAX_THREADS", "LAUNCH_BOUNDS_MIN_BLOCKS",
    "PAD_SHARED", "USE_LDG", "USE_CP_ASYNC", "USE_DOUBLE_BUFFER", "USE_WMMA",
    "WARPS_X", "WARPS_Y",
}

def strip_existing_defines(code: str) -> str:
    """
    Remove ALL #define and #undef lines for any managed parameter.
    Also removes any #ifndef / #ifdef / #endif guards wrapping them.
    Handles:
      - #define TILE_SIZE 32
      - #define TILE_SIZE  (32)
      - // #define TILE_SIZE 32   (commented out)
      - #undef TILE_SIZE
    """
    lines = code.split("\n")
    cleaned = []
    for line in lines:
        # Properly strip comment prefixes using regex
        stripped = re.sub(r'^\s*(//\s*)?', '', line)
        skip = False
        for name in _MANAGED_DEFINES:
            # Match #define NAME or #undef NAME at start of meaningful content
            if re.match(rf"#\s*(define|undef)\s+{name}\b", stripped):
                skip = True
                break
        if not skip:
            cleaned.append(line)
    return "\n".join(cleaned)


def make_individual(
    static_code    : str,
    reference_code : str,
    plan           = None,
    ind_id         = None,
    fitness        : float = -1.0,
    evaluated      : bool  = False,
    extra_defines  : dict = None,        
) -> dict:
    return {
        "id"             : ind_id or f"ind_{uuid.uuid4().hex[:8]}",
        "static_code"    : strip_existing_defines(static_code),
        "reference_code" : reference_code,
        "plan"           : plan or ExecutionPlan.random(),
        "fitness"        : fitness,
        "evaluated"      : evaluated,
        "extra_defines"  : extra_defines or {},   # store for later use
    }


DEFINE_GUARDS = """
#ifndef TILE_SIZE
#define TILE_SIZE 32
#endif
#ifndef BLOCK_SIZE
#define BLOCK_SIZE TILE_SIZE
#endif
#ifndef BLOCK_X
#define BLOCK_X 16
#endif
#ifndef BLOCK_Y
#define BLOCK_Y 16
#endif
#ifndef UNROLL
#define UNROLL 4
#endif
#ifndef VECTOR_WIDTH
#define VECTOR_WIDTH 1
#endif
#ifndef NUM_STAGES
#define NUM_STAGES 2
#endif
#ifndef USE_LAUNCH_BOUNDS
#define USE_LAUNCH_BOUNDS 0
#endif
#ifndef LAUNCH_BOUNDS_MAX_THREADS
#define LAUNCH_BOUNDS_MAX_THREADS 1024
#endif
#ifndef LAUNCH_BOUNDS_MIN_BLOCKS
#define LAUNCH_BOUNDS_MIN_BLOCKS 2
#endif
#ifndef PAD_SHARED
#define PAD_SHARED 0
#endif
#ifndef USE_LDG
#define USE_LDG 0
#endif
#ifndef USE_CP_ASYNC
#define USE_CP_ASYNC 0
#endif
#ifndef USE_DOUBLE_BUFFER
#define USE_DOUBLE_BUFFER 0
#endif
#ifndef USE_WMMA
#define USE_WMMA 0
#endif
#ifndef WARPS_X
#define WARPS_X 4
#endif
#ifndef WARPS_Y
#define WARPS_Y 2
#endif
"""

def inject_parameters(static_code: str, plan: ExecutionPlan, extra_defines: dict = None) -> str:
    clean = strip_existing_defines(static_code)
    
    # 1. Start with safety guards (provides defaults for all parameters)
    result = DEFINE_GUARDS + "\n"
    
    # 2. Add extra_defines if provided
    if extra_defines:
        extra = "\n".join(f"#define {k} {v}" for k, v in extra_defines.items())
        result += extra + "\n"
    
    # 3. Add GA-generated defines (these override/extend the guards)
    defines = plan.to_defines()
    result += defines + "\n\n"
    
    # 4. Add the clean kernel code
    result += clean
    
    return result
def tournament_select(population: list, k: int = TOURNAMENT_K) -> dict:
    contestants = random.sample(population, min(k, len(population)))
    return max(contestants, key=lambda ind: ind["fitness"])


def print_population(population: list, label: str = ""):
    if label:
        print(f"\n{'='*60}")
        print(f"  {label}")
        print(f"{'='*60}")
    for ind in sorted(population, key=lambda x: -x["fitness"]):
        status = "✓" if ind["evaluated"] else "?"
        print(f"  [{status}] {ind['id']}  fitness={ind['fitness']:.4f}  "
              f"plan={ind['plan'].to_dict()}")


## Submission To The Evaluator : 
&nbsp;&nbsp;&nbsp; In this section of the notebook, we will implement helper functions that would allow us to submit to the evaluator's script. 

In [ ]:
_PROBE_LINKER_NOISE = [
    "undefined reference to",
    "ld returned 1 exit status",
    "collect2: error",
    "undefined reference to `main'",
    "PyGILState",
    "c10::detail::torchCheck",
    "pybind11",
    "_Py_Dealloc",
]

_REAL_NVCC_ERRORS = [
    "error:",                  
    "syntax error",
    "no member named",
    "use of undeclared identifier",
    "too many arguments",
    "too few arguments",
    "expected ';'",
    "expected ')'",
    "out of registers",
    "too much global constant data",
    "ptxas error",
    "#define\""                  
]


def classify_compilation_error(error_trace: str) -> str:
    
    if error_trace is None:
        return 'unknown'
    trace_lower = error_trace.lower()

    has_real = any(pat.lower() in trace_lower for pat in _REAL_NVCC_ERRORS)
    has_noise = any(pat.lower() in trace_lower for pat in _PROBE_LINKER_NOISE)

    if has_real:#
        return 'real_error'
    if has_noise and not has_real:
        return 'probe_noise'
    return 'unknown'


def submit_to_evaluator(ind: dict) -> str:
    trial_id = ind["id"]
    op_name = ind.get("op_name", "unknown")
    matrix_size = ind["plan"].matrix_size   

    cand_code = inject_parameters(ind["static_code"], ind["plan"])
    ref_code = ind["reference_code"]

    cand_path = os.path.join(PENDING_DIR, f"{trial_id}_candidate.cu")
    ref_path = os.path.join(PENDING_DIR, f"{trial_id}_reference.cu")
    meta_path = os.path.join(PENDING_DIR, f"{trial_id}_meta.json")

    with open(cand_path, "w") as f:
        f.write(cand_code)
    with open(ref_path, "w") as f:
        f.write(ref_code)
    with open(meta_path, "w") as f:
        json.dump({"op_name": op_name, "matrix_size": matrix_size}, f)

    return trial_id


def poll_result(trial_id: str) -> dict:
   
    result_path = os.path.join(COMPLETED_DIR, f"{trial_id}_results.json")
    deadline = time.time() + POLL_TIMEOUT
    while time.time() < deadline:
        if os.path.exists(result_path):
            try:
                with open(result_path, "r") as f:
                    return json.load(f)
            except json.JSONDecodeError:
                time.sleep(POLL_INTERVAL)
                continue
        time.sleep(POLL_INTERVAL)

    return {
        "trial_id"   : trial_id,
        "status"     : "POLL_TIMEOUT",
        "reward"     : -1.0,
        "speedup"    : None,
        "error_trace": f"No result after {POLL_TIMEOUT}s",
    }

def is_valid_launch_config(plan, matrix_size):
    # 1. Block product
    if plan.block_x * plan.block_y > 1024:
        return False, f"block product {plan.block_x}×{plan.block_y} > 1024"
    # 2. Tile size vs matrix size
    if plan.tile_size > matrix_size:
        return False, f"tile_size {plan.tile_size} > matrix_size {matrix_size}"
    # 3. Grid dimensions at least 1 (already covered by tile_size <= matrix_size)
    return True, "OK"

def evaluate_individual(ind: dict) -> dict:
    if ind["evaluated"]:
        return {"status": "SKIPPED", "reward": ind["fitness"]}
    plan = ind["plan"]
    matrix_size = plan.matrix_size
    valid, reason = is_valid_launch_config(plan, matrix_size)
    if not valid:
        ind["fitness"] = -1.0
        ind["evaluated"] = True
        print(f"  [skip] {ind['id']} invalid config: {reason}")
        return {"status": "SKIP_INVALID", "reward": -1.0}
    trial_id = submit_to_evaluator(ind)
    result = poll_result(trial_id)

    status      = result.get("status", "UNKNOWN")
    reward      = result.get("reward", -1.0)
    speedup     = result.get("speedup")
    error_trace = result.get("error_trace", "")

    if status == "SUCCESS" and speedup is not None:
        ind["fitness"]   = float(speedup)
        ind["evaluated"] = True

    elif status == "WRONG_MATH":
        ind["fitness"]   = 0.0
        ind["evaluated"] = True

    elif status == "COMPILATION_ERROR":
        error_class = classify_compilation_error(error_trace)
        if error_class == 'probe_noise':
            print(f"    [eval] {ind['id']} — probe noise only (not a real nvcc error). "
                  f"Assigning fitness=0.05")
            ind["fitness"]   = 0.05
            ind["evaluated"] = True
        elif error_class == 'real_error':
            print(f"    [eval] {ind['id']} — real nvcc error. Will retry after mutation.")
            ind["fitness"]   = -1.0
            ind["evaluated"] = False   
        else:
            print(f"    [eval] {ind['id']} — unknown error type. Soft penalty.")
            ind["fitness"]   = -0.5
            ind["evaluated"] = False

    elif status == "SKIP_MISSING_KERNEL":
        ind["fitness"]   = -1.0
        ind["evaluated"] = True   

    else:
        ind["fitness"]   = -1.0
        ind["evaluated"] = False

    print(f"    {ind['id']}  status={status}  class={classify_compilation_error(error_trace) if status=='COMPILATION_ERROR' else 'n/a'}  "
          f"speedup={speedup}  fitness={ind['fitness']:.4f}")
    return result


def evaluate_population(population: list):
    pending = [ind for ind in population if not ind["evaluated"]]
    print(f"  Evaluating {len(pending)} individuals...")
    for ind in pending:
        evaluate_individual(ind)

## Population Initialization from Dataset

In [20]:
def normalize_op_name(raw_name: str) -> str:
    if not isinstance(raw_name, str):
        return "unknown"
    name_lower = raw_name.lower()
    if any(k in name_lower for k in ["matmul", "gemm", "matrix_mul", "mat_mul", "bmm", "square", "tall", "transpose"]):
        return "matmul"
    if any(k in name_lower for k in ["conv", "convolution", "depthwise", "pointwise"]):
        return "convolution"
    if "softmax" in name_lower:
        return "softmax"
    return "unknown"

In [30]:
import pandas as pd
from datasets import load_dataset

# ========== Inline Templates (no external files) ==========

MATMUL_TEMPLATE = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <c10/cuda/CUDAException.h>

// ── Tunable parameters (injected by GA) ──────────────────────────────────────
#define TILE_SIZE 32
#define BLOCK_X 16
#define BLOCK_Y 16
#define UNROLL 4
#define VECTOR_WIDTH 1
#define NUM_STAGES 2

// ── Optional flags (set by GA) ───────────────────────────────────────────────
// #define USE_LAUNCH_BOUNDS 1
// #define LAUNCH_BOUNDS_MAX_THREADS 256
// #define LAUNCH_BOUNDS_MIN_BLOCKS 2
// #define PAD_SHARED 1
// #define USE_LDG 1
// #define USE_WMMA 1

// ── WMMA path (tensor cores) ─────────────────────────────────────────────────
#if defined(USE_WMMA) && (USE_WMMA == 1)
#include <mma.h>
using namespace nvcuda::wmma;

// WMMA fragment size is fixed at 16x16x16
#define WMMA_M 16
#define WMMA_N 16
#define WMMA_K 16

// Each warp computes one WMMA_M x WMMA_N output tile.
// Block is organized as (WARPS_X * 32) threads in x, WARPS_Y threads in y.
#ifndef WARPS_X
#define WARPS_X 4
#endif
#ifndef WARPS_Y
#define WARPS_Y 2
#endif

#if defined(USE_LAUNCH_BOUNDS) && (USE_LAUNCH_BOUNDS == 1)
__launch_bounds__(LAUNCH_BOUNDS_MAX_THREADS, LAUNCH_BOUNDS_MIN_BLOCKS)
#endif
__global__ void matmul_kernel(
    const float* __restrict__ A_fp32,
    const float* __restrict__ B_fp32,
    float* __restrict__ C,
    int M, int N, int K
) {
    // Warp identity
    int warp_id   = threadIdx.x / 32;
    int warp_x    = warp_id % WARPS_X;          // which output col-tile this warp owns
    int warp_y    = warp_id / WARPS_X;          // which output row-tile this warp owns

    // Global output tile this warp is responsible for
    int row = (blockIdx.y * WARPS_Y + warp_y) * WMMA_M;
    int col = (blockIdx.x * WARPS_X + warp_x) * WMMA_N;

    // Shared memory for fp16 conversions of A and B tiles
    // Padded by 8 halfs to avoid bank conflicts
    __shared__ half As[TILE_SIZE][TILE_SIZE + 8];
    __shared__ half Bs[TILE_SIZE][TILE_SIZE + 8];

    // Accumulator fragment (fp32)
    fragment<accumulator, WMMA_M, WMMA_N, WMMA_K, float> c_frag;
    fill_fragment(c_frag, 0.0f);

    int lane = threadIdx.x % 32;

    for (int t = 0; t < (K + TILE_SIZE - 1) / TILE_SIZE; ++t) {

        // ── Cooperative load of A tile into shared memory as fp16 ────────
        // All threads in block collaborate; each loads one element
        int total = TILE_SIZE * TILE_SIZE;
        int tid   = threadIdx.x;
        for (int i = tid; i < total; i += blockDim.x) {
            int r = i / TILE_SIZE;
            int c = i % TILE_SIZE;
            int global_row = blockIdx.y * WARPS_Y * WMMA_M + r;
            int global_col = t * TILE_SIZE + c;
            float val = (global_row < M && global_col < K)
                        ? A_fp32[global_row * K + global_col] : 0.0f;
            As[r][c] = __float2half(val);
        }

        // ── Cooperative load of B tile into shared memory as fp16 ────────
        for (int i = tid; i < total; i += blockDim.x) {
            int r = i / TILE_SIZE;
            int c = i % TILE_SIZE;
            int global_row = t * TILE_SIZE + r;
            int global_col = blockIdx.x * WARPS_X * WMMA_N + c;
            float val = (global_row < K && global_col < N)
                        ? B_fp32[global_row * N + global_col] : 0.0f;
            Bs[r][c] = __float2half(val);
        }
        __syncthreads();

        // ── Each warp does TILE_SIZE/WMMA_K MMA operations ───────────────
        for (int k = 0; k < TILE_SIZE; k += WMMA_K) {
            fragment<matrix_a, WMMA_M, WMMA_N, WMMA_K, half, row_major> a_frag;
            fragment<matrix_b, WMMA_M, WMMA_N, WMMA_K, half, row_major> b_frag;

            // Row offset of this warp's A sub-tile inside shared mem
            int a_row_off = warp_y * WMMA_M;
            int b_col_off = warp_x * WMMA_N;

            load_matrix_sync(a_frag, (const half*)&As[a_row_off][k], TILE_SIZE + 8);
            load_matrix_sync(b_frag, (const half*)&Bs[k][b_col_off], TILE_SIZE + 8);
            mma_sync(c_frag, a_frag, b_frag, c_frag);
        }
        __syncthreads();
    }

    // ── Store accumulator to global memory ───────────────────────────────
    if (row < M && col < N) {
        store_matrix_sync(&C[row * N + col], c_frag, N, mem_row_major);
    }
}

torch::Tensor forward(torch::Tensor A, torch::Tensor B) {
    int M = A.size(0), K = A.size(1), N = B.size(1);
    auto C = torch::zeros({M, N}, A.options());

    // Each block covers (WARPS_Y * WMMA_M) rows and (WARPS_X * WMMA_N) cols
    dim3 threads(WARPS_X * WARPS_Y * 32);   // one warp per output tile
    dim3 blocks(
        (N + WARPS_X * WMMA_N - 1) / (WARPS_X * WMMA_N),
        (M + WARPS_Y * WMMA_M - 1) / (WARPS_Y * WMMA_M)
    );

    matmul_kernel<<<blocks, threads>>>(
        A.data_ptr<float>(), B.data_ptr<float>(), C.data_ptr<float>(), M, N, K
    );
    C10_CUDA_CHECK(cudaGetLastError());
    return C;
}

// ── FP32 tiled path (original, used when USE_WMMA is not set) ────────────────
#else

#if defined(USE_LAUNCH_BOUNDS) && (USE_LAUNCH_BOUNDS == 1)
__launch_bounds__(LAUNCH_BOUNDS_MAX_THREADS, LAUNCH_BOUNDS_MIN_BLOCKS)
#endif
__global__ void matmul_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ C,
    int M, int N, int K
) {
    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;

    #if defined(PAD_SHARED) && (PAD_SHARED == 1)
        __shared__ float As[TILE_SIZE][TILE_SIZE + 1];
        __shared__ float Bs[TILE_SIZE][TILE_SIZE + 1];
    #else
        __shared__ float As[TILE_SIZE][TILE_SIZE];
        __shared__ float Bs[TILE_SIZE][TILE_SIZE];
    #endif

    float acc = 0.0f;
    for (int t = 0; t < (K + TILE_SIZE - 1) / TILE_SIZE; ++t) {
        int a_col = t * TILE_SIZE + threadIdx.x;
        int b_row = t * TILE_SIZE + threadIdx.y;

        #if defined(USE_LDG) && (USE_LDG == 1)
            As[threadIdx.y][threadIdx.x] = (row < M && a_col < K) ? __ldg(&A[row * K + a_col]) : 0.0f;
            Bs[threadIdx.y][threadIdx.x] = (b_row < K && col < N) ? __ldg(&B[b_row * N + col]) : 0.0f;
        #else
            As[threadIdx.y][threadIdx.x] = (row < M && a_col < K) ? A[row * K + a_col] : 0.0f;
            Bs[threadIdx.y][threadIdx.x] = (b_row < K && col < N) ? B[b_row * N + col] : 0.0f;
        #endif
        __syncthreads();

        #pragma unroll UNROLL
        for (int k = 0; k < TILE_SIZE; ++k)
            acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];
        __syncthreads();
    }

    if (row < M && col < N)
        C[row * N + col] = acc;
}

torch::Tensor forward(torch::Tensor A, torch::Tensor B) {
    int M = A.size(0), K = A.size(1), N = B.size(1);
    auto C = torch::zeros({M, N}, A.options());
    dim3 blocks((N + TILE_SIZE - 1)/TILE_SIZE, (M + TILE_SIZE - 1)/TILE_SIZE);
    dim3 threads(TILE_SIZE, TILE_SIZE);
    matmul_kernel<<<blocks, threads>>>(
        A.data_ptr<float>(), B.data_ptr<float>(), C.data_ptr<float>(), M, N, K
    );
    C10_CUDA_CHECK(cudaGetLastError());
    return C;
}

#endif  // USE_WMMA

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward, "MatMul (WMMA or FP32 tiled)");
}
"""

CONV_TEMPLATE = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define TILE_SIZE 16
#define BLOCK_X 16
#define BLOCK_Y 16
#define UNROLL 4
#define VECTOR_WIDTH 1
#define NUM_STAGES 1

// Optional flags: PAD_SHARED, USE_LDG, USE_LAUNCH_BOUNDS
// Functional parameters (set via extra_defines)
// #define KERNEL_SIZE 3
// #define STRIDE 1
// #define PADDING 0
// #define DILATION 1
// #define GROUPS 1

#if defined(USE_LAUNCH_BOUNDS)
    __launch_bounds__(LAUNCH_BOUNDS_MAX_THREADS, LAUNCH_BOUNDS_MIN_BLOCKS)
#endif

__global__ void conv2d_kernel(
    const float* __restrict__ input,
    const float* __restrict__ weight,
    const float* __restrict__ bias,
    float* __restrict__ output,
    int N, int C, int H, int W,
    int C_out, int K, int stride, int padding, int dilation, int groups
) {
    int H_out = (H + 2*padding - dilation*(K-1) - 1) / stride + 1;
    int W_out = (W + 2*padding - dilation*(K-1) - 1) / stride + 1;

    int n = blockIdx.z;
    int c_out = blockIdx.y;
    int h_tile = blockIdx.x / ((W_out + BLOCK_X - 1) / BLOCK_X);
    int w_tile = blockIdx.x % ((W_out + BLOCK_X - 1) / BLOCK_X);

    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int h_out = h_tile * BLOCK_Y + ty;
    int w_out = w_tile * BLOCK_X + tx;

    if (n >= N || c_out >= C_out || h_out >= H_out || w_out >= W_out) return;

    int c_in_per_group = C / groups;
    int c_in_start = (c_out / (C_out / groups)) * c_in_per_group;
    int c_in_end = c_in_start + c_in_per_group;

    float acc = 0.0f;

    for (int c_in = c_in_start; c_in < c_in_end; ++c_in) {
        for (int ky = 0; ky < K; ++ky) {
            int h_in = h_out * stride + ky * dilation - padding;
            if (h_in < 0 || h_in >= H) continue;
            for (int kx = 0; kx < K; ++kx) {
                int w_in = w_out * stride + kx * dilation - padding;
                if (w_in < 0 || w_in >= W) continue;
                #if defined(USE_LDG) && (USE_LDG == 1)
                    float a = __ldg(&input[((n * C + c_in) * H + h_in) * W + w_in]);
                #else
                    float a = input[((n * C + c_in) * H + h_in) * W + w_in];
                #endif
                int w_idx = ((c_out * c_in_per_group + (c_in - c_in_start)) * K + ky) * K + kx;
                #if defined(USE_LDG) && (USE_LDG == 1)
                    float b = __ldg(&weight[w_idx]);
                #else
                    float b = weight[w_idx];
                #endif
                acc += a * b;
            }
        }
    }

    if (bias != nullptr) acc += bias[c_out];
    output[((n * C_out + c_out) * H_out + h_out) * W_out + w_out] = acc;
}

torch::Tensor forward(torch::Tensor input, torch::Tensor weight,
                      torch::Tensor bias, int stride, int padding,
                      int dilation, int groups) {
    // Simplified – add dimension checks as needed
    int N = input.size(0);
    int C = input.size(1);
    int H = input.size(2);
    int W = input.size(3);
    int C_out = weight.size(0);
    int K = weight.size(2);
    int H_out = (H + 2*padding - dilation*(K-1) - 1) / stride + 1;
    int W_out = (W + 2*padding - dilation*(K-1) - 1) / stride + 1;
    auto output = torch::zeros({N, C_out, H_out, W_out}, input.options());
    dim3 blocks(((W_out + BLOCK_X - 1) / BLOCK_X) * ((H_out + BLOCK_Y - 1) / BLOCK_Y), C_out, N);
    dim3 threads(BLOCK_X, BLOCK_Y);
    conv2d_kernel<<<blocks, threads>>>(
        input.data_ptr<float>(), weight.data_ptr<float>(),
        bias.defined() ? bias.data_ptr<float>() : nullptr,
        output.data_ptr<float>(),
        N, C, H, W, C_out, K, stride, padding, dilation, groups);
    return output;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward, "Conv2D");
}
"""

SOFTMAX_TEMPLATE = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>

#define BLOCK_X 128
#define UNROLL 4
#define VECTOR_WIDTH 1

#if defined(USE_LAUNCH_BOUNDS)
    __launch_bounds__(LAUNCH_BOUNDS_MAX_THREADS, LAUNCH_BOUNDS_MIN_BLOCKS)
#endif

__global__ void softmax_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int rows, int cols
) {
    int row = blockIdx.x;
    if (row >= rows) return;

    extern __shared__ float shared[];
    float* shared_max = shared;
    float* shared_sum = shared + blockDim.x;

    float max_val = -INFINITY;
    for (int i = threadIdx.x; i < cols; i += blockDim.x) {
        #if defined(USE_LDG) && (USE_LDG == 1)
            float v = __ldg(&input[row * cols + i]);
        #else
            float v = input[row * cols + i];
        #endif
        if (v > max_val) max_val = v;
    }
    shared_max[threadIdx.x] = max_val;
    __syncthreads();
    for (int s = blockDim.x/2; s > 0; s >>= 1) {
        if (threadIdx.x < s) {
            if (shared_max[threadIdx.x + s] > shared_max[threadIdx.x])
                shared_max[threadIdx.x] = shared_max[threadIdx.x + s];
        }
        __syncthreads();
    }
    max_val = shared_max[0];
    __syncthreads();

    float sum = 0.0f;
    for (int i = threadIdx.x; i < cols; i += blockDim.x) {
        float v = expf(input[row * cols + i] - max_val);
        output[row * cols + i] = v;
        sum += v;
    }
    shared_sum[threadIdx.x] = sum;
    __syncthreads();
    for (int s = blockDim.x/2; s > 0; s >>= 1) {
        if (threadIdx.x < s) {
            shared_sum[threadIdx.x] += shared_sum[threadIdx.x + s];
        }
        __syncthreads();
    }
    sum = shared_sum[0];
    __syncthreads();

    for (int i = threadIdx.x; i < cols; i += blockDim.x) {
        output[row * cols + i] /= sum;
    }
}

torch::Tensor forward(torch::Tensor input) {
    int rows = input.size(0), cols = input.size(1);
    auto output = torch::empty_like(input);
    int block_size = min(BLOCK_X, cols);
    block_size = 1 << (int)log2f(block_size);
    size_t shared_mem = 2 * block_size * sizeof(float);
    softmax_kernel<<<rows, block_size, shared_mem>>>(input.data_ptr<float>(), output.data_ptr<float>(), rows, cols);
    return output;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("forward", &forward, "Softmax");
}
"""
TEMPLATES = {
    "matmul": MATMUL_TEMPLATE,
    "convolution": CONV_TEMPLATE,
    "softmax": SOFTMAX_TEMPLATE,
}

def get_extra_defines(op_name: str, row: pd.Series) -> dict:
    extra = {}
    op_lower = op_name.lower()
    if "conv" in op_lower:
        extra["KERNEL_SIZE"] = 3
        extra["STRIDE"] = 1
        extra["PADDING"] = 0
        extra["DILATION"] = 1
        extra["GROUPS"] = 1
        if "depthwise" in op_lower:
            extra["DEPTHWISE"] = 1
    if "matmul" in op_lower:
        if "transposed_a" in op_lower or "transpose_a" in op_lower:
            extra["TRANS_A"] = 1
        if "transposed_b" in op_lower or "transpose_b" in op_lower:
            extra["TRANS_B"] = 1
        if "batch" in op_lower:
            extra["BATCHED"] = 1
    return extra

# ========== Load dataset once ==========
_DATASET_CACHE = {}

def load_dataset_split(split="level_1"):
    if "df" in _DATASET_CACHE:
        return _DATASET_CACHE["df"]
    print(f"Loading SakanaAI/AI-CUDA-Engineer-Archive split='{split}' ...")
    ds = load_dataset("SakanaAI/AI-CUDA-Engineer-Archive", split=split)
    df = ds.to_pandas()
    _DATASET_CACHE["df"] = df
    return df
    

# ========== Population Initialization (with task_id filter) ==========
def initialize_population(
    op_name_filter: str = None,   # "matmul", "convolution", "softmax"
    task_id: int = None,          # e.g., 1, 2, etc.
    matrix_size: int = 512,
    pop_size: int = POP_SIZE,
) -> list:
    """
    Seeds population from dataset rows that are Correct and have CUDA_Code.
    Filters by:
      - optional task_id (exact match on Task_ID column)
      - optional op_name_filter (after normalization)
    Always includes one baseline individual using the pure template.
    Remaining individuals are sampled from the filtered archive kernels.
    """
    df = load_dataset_split("level_1")

    # Base filter: correct, non-empty CUDA code
    mask = (
        (df["Correct"] == True) &
        (df["CUDA_Code"].notna()) &
        (df["CUDA_Code"].str.strip() != "")
    )
    filtered = df[mask].copy()

    # Filter by task_id if provided
    if task_id is not None:
        if "Task_ID" not in filtered.columns:
            raise KeyError("Column 'Task_ID' not found in dataset.")
        filtered = filtered[filtered["Task_ID"] == task_id]
        print(f"  After task_id={task_id}: {len(filtered)} rows")

    # Filter by operation name if provided
    if op_name_filter is not None:
        # Normalize operation names in filtered (create temporary column)
        filtered["_norm_op"] = filtered["Op_Name"].apply(normalize_op_name)
        filtered = filtered[filtered["_norm_op"] == op_name_filter]
        print(f"  After op='{op_name_filter}': {len(filtered)} rows")

    if len(filtered) == 0:
        raise ValueError("No kernels match the filters. Try a different task_id or op_name.")

    # Prepare baseline individual using the pure template
    op_name = op_name_filter if op_name_filter else "matmul"  # fallback
    template_code = TEMPLATES.get(op_name)
    if template_code is None:
        raise ValueError(f"No template defined for operation '{op_name}'")

    default_plan = ExecutionPlan()
    default_plan.matrix_size = matrix_size

    extra_defines = get_extra_defines(op_name, pd.Series())
    ref_code = inject_parameters(template_code, default_plan, extra_defines)

    baseline_plan = ExecutionPlan()
    baseline_plan.matrix_size = matrix_size
    baseline_ind = make_individual(
        static_code=template_code,
        reference_code=ref_code,
        plan=baseline_plan,
        ind_id=f"gen0_000_{uuid.uuid4().hex[:6]}",
        extra_defines=extra_defines,
    )
    baseline_ind["op_name"] = op_name
    baseline_ind["_source"] = "template_baseline"

    population = [baseline_ind]
    print(f"  [baseline] {baseline_ind['id']}  op={op_name}")

    # Sample remaining individuals from the archive kernels
    remaining = pop_size - 1
    if remaining <= 0:
        return population

    # Sample with replacement if needed
    sampled = filtered.sample(
        n=remaining,
        replace=(len(filtered) < remaining),
        random_state=RANDOM_SEED,
    ).reset_index(drop=True)

    for i, row in sampled.iterrows():
        raw_cuda = row["CUDA_Code"].strip()
        # Use the template for reference (same as baseline) – ensures fair comparison
        ref_code = inject_parameters(template_code, default_plan, extra_defines)

        plan = ExecutionPlan.random()
        plan.matrix_size = matrix_size

        ind_id = f"gen0_{i+1:03d}_{uuid.uuid4().hex[:6]}"
        ind = make_individual(
            static_code=raw_cuda,        # real archive kernel
            reference_code=ref_code,
            plan=plan,
            ind_id=ind_id,
            extra_defines=extra_defines,
        )
        ind["op_name"] = op_name
        ind["_source"] = "archive"
        ind["_archive_speedup"] = row.get("CUDA_Speedup_Native", None)
        population.append(ind)
        print(f"  [{i+1:03d}] {ind_id}  op={op_name}")

    print(f"\nPopulation ready: {len(population)} individuals "
          f"(1 template, {len(population)-1} from archive)")
    return population

# ========== Initialize for matmul, task_id=1 ==========
population = initialize_population(
    op_name_filter="matmul",
    task_id=1,
    matrix_size=512,
    pop_size=POP_SIZE,
)
print(f"\nPopulation size: {len(population)}")

Loading SakanaAI/AI-CUDA-Engineer-Archive split='level_1' ...
  After task_id=1: 66 rows
  After op='matmul': 66 rows
  [baseline] gen0_000_5736ec  op=matmul
  [001] gen0_001_98e1a2  op=matmul
  [002] gen0_002_1cb92b  op=matmul
  [003] gen0_003_03c8d7  op=matmul
  [004] gen0_004_718fab  op=matmul
  [005] gen0_005_c857e7  op=matmul
  [006] gen0_006_48f604  op=matmul
  [007] gen0_007_c057d4  op=matmul

Population ready: 8 individuals (1 template, 7 from archive)

Population size: 8


In [31]:
for _template_name in ("MATMUL_TEMPLATE", "CONV_TEMPLATE", "SOFTMAX_TEMPLATE"):
    if _template_name in globals():
        globals()[_template_name] = globals()[_template_name].replace(
            "#pragma unroll UNROLL",
            "#pragma unroll",
        )

In [ ]:
import random
POLL_TIMEOUT    = 300    
POLL_INTERVAL   = 2.0 
PENDING_DIR   = "pending_kernels"
COMPLETED_DIR = "completed_results"
os.makedirs(PENDING_DIR,   exist_ok=True)
os.makedirs(COMPLETED_DIR, exist_ok=True)
test_individuals = random.sample(population, min(5, len(population)))

print("Testing a few individuals with the evaluator...")
for ind in test_individuals:
    print(f"\n--- Testing {ind['id']} (op={ind.get('op_name','?')}) ---")

    result = evaluate_individual(ind)
    print(f"Result status: {result.get('status')}")
    if result.get('status') == 'SUCCESS':
        print(f"  Speedup: {result.get('speedup'):.4f}x")
    elif 'error_trace' in result:
        # Print first 500 chars of error
        print(f"  Error: {result.get('error_trace')[:500]}")

Testing a few individuals with the evaluator...

--- Testing gen0_003_03c8d7 (op=matmul) ---


## Genetic Algorithm loop : 
&nbsp;&nbsp;&nbsp; In this section of the notebook, we will start the genetic algorithm and we will report the results. 

In [ ]:
# ── History tracking (extended) ──────────────────────────────────────────────
history = []   

def record_history(gen: int, pop: list):
    evaluated = [ind for ind in pop if ind["evaluated"]]
    if not evaluated:
        return
    fitnesses = [ind["fitness"] for ind in evaluated]
    best_ind = max(evaluated, key=lambda x: x["fitness"])
    history.append({
        "generation": gen,
        "best_fitness": best_ind["fitness"],
        "mean_fitness": sum(fitnesses) / len(fitnesses),
        "best_plan": best_ind["plan"].to_dict(),
        "best_id": best_ind["id"],
        "matrix_size": best_ind["plan"].matrix_size,
        "extra_defines": best_ind.get("extra_defines", {}),
    })

record_history(0, population)

# ── Pre‑evaluation validation helper (used inside evaluate_individual) ─────
def is_valid_launch_config(plan, matrix_size):
    """Return (valid, reason)."""
    if plan.block_x * plan.block_y > 1024:
        return False, f"block product {plan.block_x}×{plan.block_y} > 1024"
    if plan.tile_size > matrix_size:
        return False, f"tile_size {plan.tile_size} > matrix_size {matrix_size}"
    # You can add shared memory check here if needed
    return True, "OK"

# ── Main GA loop ─────────────────────────────────────────────────────────────
for gen in range(1, GENERATIONS + 1):
    print(f"\n{'='*60}")
    print(f"  GENERATION {gen} / {GENERATIONS}")
    print(f"{'='*60}")

    # Sort descending by fitness
    population.sort(key=lambda x: x["fitness"], reverse=True)

    # Elitism: keep best ELITE_SIZE individuals unchanged
    elites = [copy.deepcopy(ind) for ind in population[:ELITE_SIZE]]
    new_pop = elites
    print(f"  Elites carried over: {[e['id'] for e in elites]}")

    # Use the best individual to provide static code, reference code,
    # and extra_defines (all individuals share the same operation)
    best_ind = population[0]
    reference_static = best_ind["static_code"]
    reference_code = best_ind["reference_code"]
    extra_defines = best_ind.get("extra_defines", {})
    op_name = best_ind.get("op_name", "unknown")

    # Fill the rest of the population
    while len(new_pop) < POP_SIZE:
        # Tournament selection
        p1 = tournament_select(population)
        p2 = tournament_select(population)

        # Crossover
        if random.random() < CROSSOVER_PROB:
            print(f"  [Crossover] {p1['id']} + {p2['id']} -> Qwen")
            child_plan = qwen_crossover(
                static_code=reference_static,
                parent1=p1["plan"],
                parent2=p2["plan"],
                speedup1=p1["fitness"],
                speedup2=p2["fitness"],
                op_name=op_name,
            )
        else:
            child_plan = arithmetic_crossover(p1["plan"], p2["plan"])
            print(f"  [Crossover] arithmetic {p1['id']} + {p2['id']}")

        # Mutation
        if random.random() < MUTATION_PROB:
            print(f"  [Mutation] Qwen mutating child (gen={gen})")
            child_plan = qwen_mutate(
                static_code=reference_static,
                current_plan=child_plan,
                current_speedup=max(p1["fitness"], p2["fitness"]),
                generation=gen,
                op_name=op_name,
            )

        # Validate and fix the child plan (ensure legal values)
        child_plan.validate_and_fix()

        # Create child individual, passing extra_defines from the best individual
        child = make_individual(
            static_code=reference_static,
            reference_code=reference_code,
            plan=child_plan,
            ind_id=f"gen{gen}_{len(new_pop):03d}_{uuid.uuid4().hex[:6]}",
            extra_defines=extra_defines,
        )
        child["op_name"] = op_name
        new_pop.append(child)

    population = new_pop

    # Evaluate the new population (evaluate_individual now includes cache + validation)
    print(f"\n  Evaluating generation {gen} ({len(population)} individuals)...")
    evaluate_population(population)

    # Record history and display results
    record_history(gen, population)
    print_population(population, label=f"Generation {gen} Results")

    current_best = max(population, key=lambda x: x["fitness"])
    print(f"\n  Generation {gen} best: {current_best['id']}  "
          f"fitness={current_best['fitness']:.4f}  "
          f"matrix_size={current_best['plan'].matrix_size}  "
          f"plan={current_best['plan'].to_dict()}")

print("\n" + "="*60)
print("GA COMPLETE")
print("="*60)

# Optional: Save history to CSV for later analysis
import pandas as pd
hist_df = pd.DataFrame(history)
hist_df.to_csv("ga_history.csv", index=False)
print("\nHistory saved to ga_history.csv")

In [ ]:
# ── Final best individual ──────────────────────────────────────────────────────
best = max(population, key=lambda x: x["fitness"])

print("BEST INDIVIDUAL")
print("-" * 60)
print(f"  ID      : {best['id']}")
print(f"  Fitness : {best['fitness']:.4f}x speedup")
print(f"  Plan    : {best['plan'].to_dict()}")
print()
print("FINAL KERNEL (best plan injected):")
print("-" * 60)
print(inject_parameters(best["static_code"], best["plan"]))

# ── Generation-by-generation history ─────────────────────────────────────────
print("\n\nEVOLUTION HISTORY")
print("-" * 60)
hist_df = pd.DataFrame(history)
print(hist_df[["generation", "best_fitness", "mean_fitness", "best_id"]].to_string(index=False))

# Save history
hist_df.to_json("ga_history.json", orient="records", indent=2)
print("\nHistory saved to ga_history.json")